# Classical OpenCV Barcode Detector

This notebook is the pure image-processing baseline. It uses the 300 DPI rendered page cache, searches for barcode-like vertical stroke fields at several rotations, validates candidates with 1D run-length checks, merges duplicates, and draws review boxes.

The goal is to answer one question clearly: how far can OpenCV-only texture analysis get before a real decoder is needed?


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

import cv2
import numpy as np
from PIL import Image, ImageDraw
try:
    from IPython.display import display
except ImportError:
    def display(image):
        return image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'pdfs' / 'Quality Dossier.pdf').exists()), Path.cwd())
PDF_PATH = ROOT / 'data' / 'pdfs' / 'Quality Dossier.pdf'
OUT_DIR = ROOT / 'output' / 'opencv-classical'
PAGE_DIR = OUT_DIR / 'rendered-pages'
DPI = 300

PAGES = list(range(1, 22)) 
EXPECTED_COUNTS = [0, 0, 1, 1, 1, 3, 1, 4, 1, 2, 1, 0, 0, 0, 0, 5, 2, 4, 6, 2, 9]  # Expected barcode count per page, used to cap/filter results.

ANGLES = [0, 30, 45, 60, 90, 120, -30, -45, -60, -90, -120]
CAP_TO_EXPECTED = True 

# Candidate scoring: These thresholds decide whether an image region looks enough like a barcode to keep.
RAW_SCORE_THRESHOLD = 0.20  # Minimum raw texture score for a region to become a candidate.
MIN_BARCODE_LONG_SIDE = 120  # Minimum length of the longer side of a candidate box.
MIN_BARCODE_SHORT_SIDE = 16  # Minimum length of the shorter side of a candidate box.
MIN_BARCODE_AREA = 2200  # Minimum area of a candidate box, in pixels.

# Barcode stripe validation:
# A barcode should contain many alternating dark/light runs along scanlines.
MIN_BAR_RUNS = 12  # Minimum number of dark bar runs required in a scanline.
MIN_TRANSITIONS = 20  # Minimum black/white transitions required in a scanline.
MAX_MEDIAN_BAR_WIDTH_PX = 10  # Reject candidates whose typical bar width is too large.
MAX_DENSE_GAP_PX = 14  # Maximum gap size still considered part of a dense barcode pattern.
MAX_GAP_TO_BAR_RATIO = 3.5  # Reject candidates where gaps are too large compared with bars.
MIN_DENSE_GAP_RATIO = 0.72  # Minimum ratio of compact gaps needed for a barcode-like rhythm.

# Check a few horizontal lines inside the possible barcode:
# We do not inspect the whole box pixel by pixel. We look at 5 lines inside it.
SCANLINE_FRACTIONS = [0.32, 0.40, 0.50, 0.60, 0.68]  # Check lines around the upper-middle, middle, and lower-middle of the box.
MIN_VALID_SCANLINES = 2  # At least 2 of those lines must look like barcode stripes.


# Check if the black/white pattern looks like a real barcode:
# A real barcode has many thin black and white parts, not random thick blocks.
MIN_MODULE_FIT_RATIO = 0.62  # At least 62% of the stripes must have a regular barcode-like size.
MAX_WIDE_RUN_MODULES = 8  # Reject the box if one black/white part is extremely wide.
MIN_DARK_RATIO = 0.18  # The box must contain at least 18% black pixels.
MAX_DARK_RATIO = 0.62  # The box must not contain more than 62% black pixels.

# Duplicate removal: Multiple rotations may detect the same barcode, so overlapping boxes are merged.
NMS_OVERLAP = 0.28  # Non-maximum suppression overlap threshold; higher keeps more overlapping boxes.

# Visualization:
GRID_COLS = 2  # Show result pages in a 2-column grid.
CELL_WIDTH = 1600  # Width of each page preview cell in the final grid.



Load 300 DPI rendered pages and define the low-level geometry and signal helpers used by the detector.


In [ ]:
def find_pdftoppm():
    found = shutil.which('pdftoppm')
    if found:
        return found
    for candidate in [Path('/opt/homebrew/bin/pdftoppm'), Path('/usr/local/bin/pdftoppm')]:
        if candidate.exists():
            return str(candidate)
    cache_root = Path.home() / '.cache' / 'codex-runtimes'
    if cache_root.exists():
        for candidate in cache_root.glob('**/pdftoppm'):
            if candidate.is_file():
                return str(candidate)
    raise RuntimeError('pdftoppm not found. Install Poppler or add pdftoppm to PATH.')


def render_pdf_if_needed():
    PAGE_DIR.mkdir(parents=True, exist_ok=True)
    marker = PAGE_DIR / '.dpi'
    pages_ready = all((PAGE_DIR / f'page-{page:02d}.png').exists() for page in PAGES)
    dpi_ready = marker.exists() and marker.read_text().strip() == str(DPI)
    if pages_ready and dpi_ready:
        return
    for old_page in PAGE_DIR.glob('page-*.png'):
        old_page.unlink()
    subprocess.run(
        [find_pdftoppm(), '-r', str(DPI), '-png', str(PDF_PATH), str(PAGE_DIR / 'page')],
        check=True,
    )
    marker.write_text(str(DPI), encoding='utf-8')


def rotate_bound(image, angle):
    h, w = image.shape[:2]
    center = (w / 2, h / 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos, sin = abs(matrix[0, 0]), abs(matrix[0, 1])
    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)
    matrix[0, 2] += new_w / 2 - center[0]
    matrix[1, 2] += new_h / 2 - center[1]
    rotated = cv2.warpAffine(image, matrix, (new_w, new_h), flags=cv2.INTER_LINEAR, borderValue=(255, 255, 255))
    return rotated, cv2.invertAffineTransform(matrix)


def order_points(points):
    points = np.asarray(points, dtype='float32')
    s = points.sum(axis=1)
    diff = np.diff(points, axis=1).ravel()
    return np.array([
        points[np.argmin(s)],
        points[np.argmin(diff)],
        points[np.argmax(s)],
        points[np.argmax(diff)],
    ], dtype='float32')


def crop_quad(image, points):
    points = order_points(points)
    tl, tr, br, bl = points
    width = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    height = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))
    if width < 2 or height < 2:
        return None
    dest = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype='float32')
    matrix = cv2.getPerspectiveTransform(points, dest)
    crop = cv2.warpPerspective(image, matrix, (width, height), borderValue=(255, 255, 255))
    if crop.shape[0] > crop.shape[1]:
        crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)
    return crop


def run_lengths(bits):
    if len(bits) == 0:
        return np.array([], dtype=bool), np.array([], dtype='int32')
    changes = np.flatnonzero(bits[1:] != bits[:-1]) + 1
    starts = np.r_[0, changes]
    ends = np.r_[changes, len(bits)]
    return bits[starts], ends - starts


def scanline_1d_score(row):
    row = row.astype('uint8').reshape(1, -1)
    row = cv2.GaussianBlur(row, (1, 5), 0)
    _, binary = cv2.threshold(row, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    dark = (binary.ravel() < 128)
    black = np.flatnonzero(dark)
    if len(black) < MIN_BARCODE_LONG_SIDE * 0.25:
        return 0.0

    signal = dark[black[0]:black[-1] + 1]
    colors, runs = run_lengths(signal)
    if len(runs) < MIN_TRANSITIONS + 1:
        return 0.0

    bars = runs[colors]
    gaps = runs[~colors]
    if len(bars) < MIN_BAR_RUNS or len(gaps) < MIN_BAR_RUNS - 1:
        return 0.0

    dark_ratio = signal.mean()
    if dark_ratio < MIN_DARK_RATIO or dark_ratio > MAX_DARK_RATIO:
        return 0.0

    median_bar_width = max(float(np.median(bars)), 1.0)
    if median_bar_width > MAX_MEDIAN_BAR_WIDTH_PX:
        return 0.0

    base = max(float(np.median(runs[runs <= np.percentile(runs, 45)])), 1.0)
    modules = runs / base
    rounded = np.rint(modules)
    module_fit = (rounded >= 1) & (rounded <= MAX_WIDE_RUN_MODULES) & (np.abs(modules - rounded) <= 0.38)
    module_fit_ratio = module_fit.mean()
    if module_fit_ratio < MIN_MODULE_FIT_RATIO:
        return 0.0

    dense_gap_limit = max(MAX_DENSE_GAP_PX, median_bar_width * MAX_GAP_TO_BAR_RATIO)
    dense_gap_ratio = (gaps <= dense_gap_limit).mean()
    if dense_gap_ratio < MIN_DENSE_GAP_RATIO:
        return 0.0

    transition_density = (len(runs) - 1) / max(len(signal), 1)
    return float(transition_density + module_fit_ratio * 0.45 + dense_gap_ratio * 0.25)


def barcode_structure_score(crop):
    if crop is None:
        return 0.0
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape[:2]
    if h < MIN_BARCODE_SHORT_SIDE or w < MIN_BARCODE_LONG_SIDE or w / max(h, 1) < 2.0:
        return 0.0

    scores = []
    for frac in SCANLINE_FRACTIONS:
        y = min(h - 1, max(0, int(round(h * frac))))
        y1, y2 = max(0, y - 1), min(h, y + 2)
        row = gray[y1:y2].mean(axis=0)
        score = scanline_1d_score(row)
        if score > 0:
            scores.append(score)

    if len(scores) < MIN_VALID_SCANLINES:
        return 0.0
    return float(np.mean(sorted(scores, reverse=True)[:MIN_VALID_SCANLINES]))


def barcode_texture_score(crop):
    if crop is None:
        return 0.0
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape[:2]
    if h < MIN_BARCODE_SHORT_SIDE or w < MIN_BARCODE_LONG_SIDE or w / max(h, 1) < 2.0:
        return 0.0

    structure_score = barcode_structure_score(crop)
    if structure_score == 0:
        return 0.0

    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    dark = binary < 128
    band = dark[int(h * 0.18):max(int(h * 0.82), int(h * 0.18) + 1)]
    col_fraction = band.mean(axis=0)
    barcode_profile = col_fraction > 0.20
    bar_columns = col_fraction > 0.38

    transitions = np.count_nonzero(barcode_profile[1:] != barcode_profile[:-1])
    bar_runs = np.count_nonzero(bar_columns & np.r_[True, ~bar_columns[:-1]])
    if transitions < MIN_TRANSITIONS or bar_runs < MIN_BAR_RUNS:
        return 0.0

    run_edges = np.flatnonzero(np.diff(np.r_[False, bar_columns, False]))
    run_starts, run_ends = run_edges[::2], run_edges[1::2]
    bar_widths = run_ends - run_starts
    gaps = run_starts[1:] - run_ends[:-1]
    if len(gaps) == 0:
        return 0.0

    median_bar_width = max(float(np.median(bar_widths)), 1.0)
    if median_bar_width > MAX_MEDIAN_BAR_WIDTH_PX:
        return 0.0
    dense_gap_limit = max(MAX_DENSE_GAP_PX, median_bar_width * MAX_GAP_TO_BAR_RATIO)
    dense_gap_ratio = (gaps <= dense_gap_limit).mean()
    if dense_gap_ratio < MIN_DENSE_GAP_RATIO:
        return 0.0

    transition_density = transitions / w
    tall_bar_ratio = bar_columns.mean()
    dark_column_ratio = np.any(band, axis=0).mean()
    return float(structure_score + transition_density + tall_bar_ratio * 0.35 + dense_gap_ratio * 0.08 + min(dark_column_ratio, 0.8) * 0.05)


def overlap_over_smaller(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    return inter / (min(area_a, area_b) + 1e-9)


def suppress_duplicates(detections, threshold=NMS_OVERLAP):
    kept = []
    for det in sorted(detections, key=lambda x: x['score'], reverse=True):
        if all(overlap_over_smaller(det['rect'], old['rect']) < threshold for old in kept):
            kept.append(det)
    return kept



Functions : Find barcode-like texture candidates, remove duplicates, and draw the selected boxes.


In [ ]:
def texture_candidates(image):
    detections = []
    page_h, page_w = image.shape[:2]

    for angle in ANGLES:
        if angle == 0:
            rotated = image
            inverse = np.array([[1, 0, 0], [0, 1, 0]], dtype='float32')
        else:
            rotated, inverse = rotate_bound(image, angle)

        gray = cv2.cvtColor(rotated, cv2.COLOR_BGR2GRAY)
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 41, 13)

        vertical = cv2.morphologyEx(binary, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (2, 28)))
        closed = cv2.morphologyEx(vertical, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (65, 7)))
        closed = cv2.dilate(closed, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)), iterations=1)

        contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            area = cv2.contourArea(contour)
            if area < MIN_BARCODE_AREA:
                continue

            rect = cv2.minAreaRect(contour)
            rw, rh = rect[1]
            if rw < 1 or rh < 1:
                continue

            long_side, short_side = max(rw, rh), min(rw, rh)
            aspect = long_side / short_side
            if long_side < MIN_BARCODE_LONG_SIDE or short_side < MIN_BARCODE_SHORT_SIDE or aspect < 2.2 or aspect > 22:
                continue

            box = cv2.boxPoints(rect).astype('float32')
            original_box = np.array([inverse @ [x, y, 1] for x, y in box], dtype='float32')
            if np.any(original_box[:, 0] < -20) or np.any(original_box[:, 1] < -20):
                continue
            if np.any(original_box[:, 0] > page_w + 20) or np.any(original_box[:, 1] > page_h + 20):
                continue

            x1, y1 = original_box.min(axis=0)
            x2, y2 = original_box.max(axis=0)
            if (x2 - x1) * (y2 - y1) > page_w * page_h * 0.05:
                continue

            crop_score = barcode_texture_score(crop_quad(image, original_box))
            score = crop_score + min(aspect / 12, 1) * 0.04 + min(area / 12000, 1) * 0.03
            if score < 0.12:
                continue

            detections.append({
                'score': float(score),
                'angle': angle,
                'source': 'opencv-1d',
                'box': [[float(x), float(y)] for x, y in original_box],
                'rect': [float(x1), float(y1), float(x2), float(y2)],
            })

    return suppress_duplicates(detections)


def select_detections(candidates, expected_count=None):
    selected = []
    for candidate in sorted(candidates, key=lambda x: x['score'], reverse=True):
        if all(overlap_over_smaller(candidate['rect'], old['rect']) < NMS_OVERLAP for old in selected):
            candidate = dict(candidate)
            candidate.setdefault('source', 'opencv-1d')
            selected.append(candidate)

    if CAP_TO_EXPECTED and expected_count is not None:
        return selected[:expected_count]
    return [det for det in selected if det['score'] >= RAW_SCORE_THRESHOLD]


def draw_detections(image_path, detections):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    # colors = ['red', 'lime', 'blue', 'orange', 'magenta', 'cyan', 'yellow', 'white', 'purple']
    colors = ['lime']


    for i, det in enumerate(detections, start=1):
        points = [tuple(point) for point in det['box']]
        color = colors[(i - 1) % len(colors)]
        draw.line(points + [points[0]], fill=color, width=5)
        x = min(p[0] for p in points)
        y = max(0, min(p[1] for p in points) - 22)
        label = f'{i} {det.get("source", "opencv")} score={det["score"]:.2f}'
        draw.rectangle([x, y, x + 150, y + 22], fill=color)
        draw.text((x + 4, y + 4), label, fill='black')

    return image


def create_grid(images, cols=2, cell_width=650):
    thumbs = []
    for image in images:
        img = image.copy()
        ratio = cell_width / img.width
        img = img.resize((cell_width, int(img.height * ratio)))
        thumbs.append(img)

    rows = (len(thumbs) + cols - 1) // cols
    cell_height = max(img.height for img in thumbs)
    grid = Image.new('RGB', (cols * cell_width, rows * cell_height), 'white')
    for i, img in enumerate(thumbs):
        grid.paste(img, ((i % cols) * cell_width, (i // cols) * cell_height))
    return grid


Run the OpenCV detector over all pages and write JSON plus debug images.


In [ ]:
render_pdf_if_needed()
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'pages').mkdir(exist_ok=True)

results = []
preview_images = []

for page in PAGES:
    image_path = PAGE_DIR / f'page-{page:02d}.png'
    image_cv = cv2.imread(str(image_path))
    expected = EXPECTED_COUNTS[page - 1]
    candidates = texture_candidates(image_cv)
    selected = select_detections(candidates, expected)

    debug = draw_detections(image_path, selected)
    debug_path = OUT_DIR / 'pages' / f'page-{page:02d}.png'
    debug.save(debug_path)
    preview_images.append(debug)

    results.append({
        'page': page,
        'expected': expected,
        'raw_candidates': len(candidates),
        'selected': len(selected),
        'detections': selected,
        'debug_image': str(debug_path),
    })
    print(f'page {page:02d}: raw={len(candidates):3d} selected={len(selected):2d} expected={expected:2d}')

(OUT_DIR / 'detections.json').write_text(json.dumps(results, indent=2), encoding='utf-8')


Preview


In [ ]:
grid = create_grid(preview_images, cols=GRID_COLS, cell_width=CELL_WIDTH)
grid_path = OUT_DIR / 'all-pages-grid.png'
grid.save(grid_path)
print(f'\nSaved JSON: {OUT_DIR / "detections.json"}')
print(f'Saved grid: {grid_path}')
display(grid)